# Bernoulli Naive Bayes - Day 4 (From Scratch)

## What is Bernoulli Naive Bayes?

Bernoulli Naive Bayes is a subcategory of the Naive Bayes Algorithm. It is typically used when the data is binary and it models the occurrence of features using Bernoulli distribution. It is used for the classification of binary features such as 'Yes' or 'No', '1' or '0', 'True' or 'False' etc. Features are independent of one another.

---

## Mathematics

In Bernoulli Naive Bayes, each feature is conditionally independent given the class y:

$$p(x_i | y) = p(i|y)^{x_i} + (1 - p(i|y))(1 - x_i)$$

Where:
- $p(x_i | y)$ is the conditional probability of $x_i$ occurring provided y has occurred
- $i$ is the feature index
- $x_i$ holds binary value either 0 or 1

---

## Bernoulli Distribution

$$f(x) = \begin{cases} p^x \cdot (1-p)^{1-x} & \text{if x=0,1} \\ 0 & \text{otherwise} \end{cases}$$

Where:
- $x=1$: f(x) = p (success)
- $x=0$: f(x) = 1-p (failure)
- $p$ denotes the probability of success

---

## Laplace Smoothing

$$P(w_i = 1 | C) = \frac{count(w_i, C) + 1}{N_C + 2}$$

Where $N_C$ is the number of documents in class C.

---

## Comparison of Naive Bayes Models

| Aspect | Gaussian NB | Multinomial NB | Bernoulli NB |
|--------|-------------|----------------|--------------|
| Feature Type | Continuous | Discrete (counts) | Binary (presence/absence) |
| Assumption | Gaussian distribution | Multinomial distribution | Bernoulli distribution |
| Use Case | Height, weight, measurements | Text classification (word counts) | Spam detection, binary features |
| Data Representation | Continuous variables | Discrete counts/frequencies | Binary (0 or 1) |

---

## One-Line Summary

**Bernoulli Naive Bayes handles binary features using Bernoulli distribution and Laplace smoothing for classification.**

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter

print("="*50)
print("BERNOULLI NAIVE BAYES DAY 4 - FROM SCRATCH")
print("="*50)

## Bernoulli Naive Bayes Implementation from Scratch

In [ ]:
class BernoulliNaiveBayes:
    """
    Bernoulli Naive Bayes Classifier from Scratch
    """
    
    def __init__(self, alpha=1.0):
        self.alpha = alpha  # Laplace smoothing parameter
        self.classes = None
        self.class_priors = {}
        self.feature_probs = {}  # P(feature=1 | class)
    
    def fit(self, X, y):
        """
        Train the Bernoulli Naive Bayes classifier
        """
        self.classes = np.unique(y)
        n_samples = len(y)
        
        # Calculate class priors
        for cls in self.classes:
            self.class_priors[cls] = np.sum(y == cls) / n_samples
        
        # Calculate feature probabilities for each class
        for cls in self.classes:
            X_cls = X[y == cls]
            n_cls = len(X_cls)
            
            # Count occurrences of each feature in class
            feature_counts = np.sum(X_cls, axis=0)
            
            # Laplace smoothing: (count + alpha) / (n_cls + 2*alpha)
            self.feature_probs[cls] = (feature_counts + self.alpha) / (n_cls + 2 * self.alpha)
    
    def predict_single(self, x):
        """
        Predict a single sample
        """
        posteriors = {}
        
        for cls in self.classes:
            # Start with log prior
            posterior = np.log(self.class_priors[cls])
            
            # Multiply by Bernoulli likelihoods
            for i in range(len(x)):
                p = self.feature_probs[cls][i]
                if x[i] == 1:
                    posterior += np.log(p)
                else:
                    posterior += np.log(1 - p)
            
            posteriors[cls] = posterior
        
        return max(posteriors, key=posteriors.get)
    
    def predict(self, X):
        return np.array([self.predict_single(x) for x in X])
    
    def score(self, X, y):
        return np.mean(self.predict(X) == y)

## Example: Spam Detection with Binary Features

In [ ]:
# Training data
data = {
    'Message': [
        'buy cheap now',
        'limited offer buy',
        'meet me now',
        'lets catch up'
    ],
    'Class': ['Spam', 'Spam', 'Not Spam', 'Not Spam']
}

df = pd.DataFrame(data)
print("Training Dataset:")
print(df)

In [ ]:
# Build vocabulary
def build_vocabulary(texts):
    vocabulary = set()
    for text in texts:
        words = text.lower().split()
        vocabulary.update(words)
    return sorted(list(vocabulary))

vocabulary = build_vocabulary(df['Message'])
print(f"Vocabulary size: {len(vocabulary)}")
print("Vocabulary:", vocabulary)

In [ ]:
# Convert text to binary feature vectors
def text_to_binary_vector(text, vocabulary):
    words = text.lower().split()
    return np.array([1 if word in words else 0 for word in vocabulary])

X = np.array([text_to_binary_vector(text, vocabulary) for text in df['Message']])
y = np.array([1 if label == 'Spam' else 0 for label in df['Class']])

print("\nBinary Feature Matrix:")
print(pd.DataFrame(X, columns=vocabulary))
print("\nTarget labels:", y)

In [ ]:
# Train Bernoulli Naive Bayes
bnb = BernoulliNaiveBayes(alpha=1.0)
bnb.fit(X, y)

print("\nClass Priors:")
for cls in bnb.classes:
    class_name = "Spam" if cls == 1 else "Not Spam"
    print(f"P({class_name}) = {bnb.class_priors[cls]:.4f}")

print("\nFeature Probabilities P(feature=1 | class):")
for cls in bnb.classes:
    class_name = "Spam" if cls == 1 else "Not Spam"
    print(f"\n{class_name}:")
    for i, word in enumerate(vocabulary):
        print(f"  P({word}=1) = {bnb.feature_probs[cls][i]:.4f}")

In [ ]:
# Test message
test_message = "buy now"
test_vector = text_to_binary_vector(test_message, vocabulary)
prediction = bnb.predict_single(test_vector)

print("\n" + "="*40)
print(f"Test Message: '{test_message}'")
print(f"Binary Vector: {test_vector}")
print(f"Prediction: {'Spam' if prediction == 1 else 'Not Spam'}")

In [ ]:
# Test multiple messages
test_messages = [
    "buy now",
    "meet me",
    "limited offer",
    "lets catch up"
]

print("\n" + "="*40)
print("Multiple Test Messages:")
print("="*40)

for msg in test_messages:
    vec = text_to_binary_vector(msg, vocabulary)
    pred = bnb.predict_single(vec)
    print(f"'{msg:<15}' -> {'Spam' if pred == 1 else 'Not Spam'}")

In [ ]:
# Day 4 Completed
print("\n" + "="*50)
print("BERNOULLI NAIVE BAYES DAY 4 COMPLETED")
print("="*50)
print("Topics covered:")
print("- Bernoulli Naive Bayes definition")
print("- Bernoulli distribution formula")
print("- Laplace smoothing for binary features")
print("- Spam detection with binary features")
print("- Text to binary vector conversion")
print("- Comparison of Naive Bayes models")
print("="*50)"
print("\n*** NAIVE BAYES MODULE COMPLETED ***")
print("\nComplete Naive Bayes Series:")
print("Day 1: Introduction to Naive Bayes")
print("Day 2: Gaussian Naive Bayes")
print("Day 3: Multinomial Naive Bayes")
print("Day 4: Bernoulli Naive Bayes (Final)")